In [1]:
import subprocess
import os
import pandas as pd
import netCDF4
import numpy as np
import glob
import time
import matplotlib.pyplot as plt
import copy
import xarray as xr
from datetime import datetime, timedelta 
import dask
from scipy.interpolate import griddata
#from ocean_c_lab_tools import *
#from celluloid import Camera 
#import PyCO2SYS as csys
import seawater as sw
from roms_regrid import *

/tmp/ipykernel_3409921/3005166470.py:17: UserWarning: The seawater library is deprecated! Please use gsw instead.
  import seawater as sw


In [2]:
#HAFRO_path='/home/x-uheede/R/HAFRO/Hafro_cruises.xls'
model_grid_path="/home/x-uheede/S/Iceland3_MARBL_2024_60m/P_INPUT/Iceland3_grid_MAT.nc"
# Grid parameters, only modify these if grid is made in MATLAB
vert_levels=60
theta_s_model=5
theta_b_model=2
hc_model=300
model_bgc_dia_path="/home/x-uheede/S/Iceland3_MARBL_2024_60m_CDR/Iceland3_MARBL_2024_bgc_dia.2024022[0-4]??????.nc"
model_bgc_path="/home/x-uheede/S/Iceland3_MARBL_2024_60m_CDR/Iceland3_MARBL_2024_bgc.20240220??????.nc"

target_depth_levels=[1,2,3,4,5,7,9,10,12,14,15,16,18,20,26,30,36,40,50,80] # Specify depth levels of interest
thinner=5 # specify the temporal frequency of data being read (i.e. no need to read in hourly data)


In [3]:
from roms_tools import Grid, ROMSOutput

In [4]:
grid = Grid.from_file(
    model_grid_path
)

2026-01-27 18:17:18 - WARNING - Vertical coordinates (Cs_r, Cs_w) not found in grid file.
2026-01-27 18:17:18 - INFO - === Preparing the vertical coordinate system using N = 100, theta_s = 5.0, theta_b = 2.0, hc = 300.0 ===
2026-01-27 18:17:18 - INFO - Total time: 0.004 seconds
2026-01-27 18:17:18 - INFO - ================================================================================================


In [5]:
#Only run this cell if grid is made in MATLAB
grid.update_vertical_coordinate(N=vert_levels, theta_s=theta_s_model, theta_b=theta_b_model, hc=hc_model, verbose=False)

In [6]:
import xarray as xr
import numpy as np

# Load ROMS output using your pattern
roms_output = ROMSOutput(
    grid=grid,
    path=[
        model_bgc_dia_path,
    ],
    use_dask=True,
)

ds = roms_output.regrid(depth_levels=target_depth_levels)


In [ ]:
ds.load()

In [ ]:
# Maximum pH at each grid cell (over time)
ph_max = ds["pH_3D"].max(dim="time", skipna=True).max(dim="depth", skipna=True)
ph_max.name = "PH_max"
ph_max.attrs["long_name"] = "Maximum pH"


In [ ]:
# Land mask: 1 = land, 0 = ocean
land_mask = xr.where(np.isnan(ph_max), 1, 0)
land_mask.name = "land_mask"

In [ ]:
from pyproj import Geod
import matplotlib.pyplot as plt

def add_scalebar(ax, length_km, location=(0.1, 0.05), linewidth=3):
    """
    Add a scale bar to a Cartopy map.

    Parameters
    ----------
    ax : cartopy.mpl.geoaxes.GeoAxes
        Map axes
    length_km : float
        Length of scale bar in kilometers
    location : tuple
        (x_frac, y_frac) position in axes coordinates
    linewidth : int
        Line width
    """

    geod = Geod(ellps="WGS84")

    # Get current map extent in PlateCarree
    lon_min, lon_max, lat_min, lat_max = ax.get_extent(ccrs.PlateCarree())

    # Starting point
    lon_start = lon_min + location[0] * (lon_max - lon_min)
    lat_start = lat_min + location[1] * (lat_max - lat_min)

    # Compute endpoint using geodesic
    lon_end, lat_end, _ = geod.fwd(
        lon_start, lat_start,
        az=90,              # eastward
        dist=length_km * 1000
    )

    # Draw scale bar
    ax.plot(
        [lon_start, lon_end],
        [lat_start, lat_start],
        transform=ccrs.PlateCarree(),
        color="black",
        linewidth=linewidth
    )

    # Label
    ax.text(
        (lon_start + lon_end) / 2,
        lat_start + 0.002,
        f"{length_km} km",
        transform=ccrs.PlateCarree(),
        ha="center",
        va="bottom",
        fontsize=9
    )


In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

ph_max_plot = ph_max.assign_coords(
    lon=((ph_max.lon + 180) % 360) - 180
).sortby("lon")

land_mask_plot = land_mask.assign_coords(
    lon=((land_mask.lon + 180) % 360) - 180
).sortby("lon")


extent_upper_hval = [
    -21.6,   # lon min
    -21.37,  # lon max
    64.34,   # lat min
    64.42    # lat max
]
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

fig = plt.figure(figsize=(8, 8))
ax = plt.axes(projection=ccrs.Mercator())

ax.set_extent(extent_upper_hval, crs=ccrs.PlateCarree())

# --------------------------------------
# Land mask (plotted FIRST, underneath)
# --------------------------------------
ax.contourf(
    land_mask_plot.lon,
    land_mask_plot.lat,
    land_mask_plot,
    levels=[0.5, 1.5],
    colors=["lightgray"],
    transform=ccrs.PlateCarree(),
    zorder=0
)

# --------------------------------------
# Maximum pH (ocean only)
# --------------------------------------
cf = ax.contourf(
    ph_max_plot.lon,
    ph_max_plot.lat,
    ph_max_plot,
    levels=40, vmin=7.9, vmax=9,
    cmap="viridis",
    transform=ccrs.PlateCarree(),
    zorder=1
)
# Mark Pier
ax.plot(
    -21.465904, 64.394213,
    marker="*", color="red", markersize=6,
    transform=ccrs.PlateCarree()
)
ax.text(
    -21.465904, 64.394213+0.005,
    "Pier",
    color="red",
    fontsize=9,
    transform=ccrs.PlateCarree(),
    ha="left",
    va="bottom"
)


# Gridlines only (no coastlines)
gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5)
gl.top_labels = False
gl.right_labels = False

# Colorbar
cbar = plt.colorbar(cf, ax=ax, shrink=0.75, pad=0.05)
cbar.set_label("Maximum pH")

ax.set_title("Maximum pH (upper Hvalfjörður)")

add_scalebar(
    ax,
    length_km=1,        # good for upper Hvalfjörður
    location=(0.1, 0.08)
)
plt.tight_layout()
plt.show()



In [ ]:
# Maximum pH over the spatial domain for each time step
pH_max_time = ds["pH_3D"].max(dim=("lat", "lon", "depth"), skipna=True)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))
plt.plot(
    pH_max_time["time"],
    pH_max_time,
    color="darkblue",
    linewidth=2
)

plt.title("Maximum pH over domain (per time step)")
plt.xlabel("Time")
plt.ylabel("pH")
plt.grid(True)
plt.ylim(8,10.7)
plt.tight_layout()
plt.show()


In [ ]:
# Maximum pH at each depth over all time and space
pH_max_profile = ds["pH_3D"].max(
    dim=("time", "lat", "lon"),
    skipna=True
)

In [ ]:
import matplotlib.pyplot as plt

depth = ds["depth"]

plt.figure(figsize=(5, 7))

plt.plot(
    pH_max_profile,
    depth,
    linewidth=2,
    color="darkblue",
    label="Max pH"
)

plt.gca().invert_yaxis()
plt.xlabel("pH")
plt.ylabel("Depth (m)")
plt.title("Maximum pH profile\n(max over time, lat, lon)")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
import xarray as xr
import numpy as np
target_depth_levels=[1,2,3,4,5,7,9,10,12,14,15] # Specify depth levels of interest

# Load ROMS output using your pattern
roms_output = ROMSOutput(
    grid=grid,
    path=[
        model_bgc_path,
    ],
    use_dask=True,
)

ds_trac = roms_output.regrid(depth_levels=target_depth_levels, var_names=["ALK", "ALK_ALT_CO2"])


In [ ]:
ds_trac.load()

In [ ]:
pip install pyco2sys

In [ ]:
import xarray as xr
import numpy as np
import PyCO2SYS as pyco2

# ==================================================
# USER-SPECIFIED BACKGROUND STATE
# ==================================================
BACKGROUND = dict(
    ALK = 2300.0,     # umol/kg
    DIC = 2150.0,     # umol/kg
    T = 5.0,          # deg C
    S = 35.0,         # PSU
    P = 0.0,          # dbar
    PO4 = 0.0,        # umol/kg
    Si = 0.0          # umol/kg
)

# ==================================================
# Open dataset
# ==================================================
ds = ds_trac

# ==================================================
# Compute alkalinity anomaly
# ==================================================
dALK = ds["ALK"] - ds["ALK_ALT_CO2"]

# (optional) choose surface layer
dALK_surf = dALK.isel(depth=0)

# ==================================================
# Collapse to maximum anomaly in space & time
# (worst-case pH excursion)
# ==================================================
dALK_max = float(dALK.max(skipna=True))

print(f"Maximum alkalinity anomaly: {dALK_max:.2f} µmol/kg")

# ==================================================
# Carbonate chemistry: background pH
# ==================================================
bg = pyco2.sys(
    par1=BACKGROUND["ALK"],
    par2=BACKGROUND["DIC"],
    par1_type=1,      # ALK
    par2_type=2,      # DIC
    salinity=BACKGROUND["S"],
    temperature=BACKGROUND["T"],
    pressure=BACKGROUND["P"],
    total_phosphate=BACKGROUND["PO4"],
    total_silicate=BACKGROUND["Si"]
)

pH_bg = bg["pH_total"]

# ==================================================
# Perturbed state (ALK + anomaly)
# ==================================================
pert = pyco2.sys(
    par1=BACKGROUND["ALK"] + dALK_max,
    par2=BACKGROUND["DIC"],
    par1_type=1,
    par2_type=2,
    salinity=BACKGROUND["S"],
    temperature=BACKGROUND["T"],
    pressure=BACKGROUND["P"],
    total_phosphate=BACKGROUND["PO4"],
    total_silicate=BACKGROUND["Si"]
)

pH_pert = pert["pH_total"]

# ==================================================
# pH excursion
# ==================================================
dpH = pH_pert - pH_bg

print("\n===== RESULTS =====")
print(f"Background pH        : {pH_bg:.4f}")
print(f"Perturbed pH         : {pH_pert:.4f}")
print(f"Maximum ΔpH excursion: {dpH:+.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import PyCO2SYS as pyco2

# ==================================================
# USER CONSTANTS
# ==================================================
T = 5.0        # deg C
S = 34.5       # PSU
P = 0.0        # dbar
PO4 = 0.9
Si = 7.6

# ==================================================
# Background phase space
# ==================================================
ALK_vals = np.linspace(2250, 2350, 10)   # µmol/kg
DIC_vals = np.linspace(2000, 2200, 10)   # µmol/kg

# ==================================================
# Dataset + maximum alkalinity anomaly
# ==================================================
ds = ds_trac
dALK = ds["ALK"] - ds["ALK_ALT_CO2"]
dALK_max = float(dALK.max(skipna=True))

print(f"Using maximum ALK anomaly: {dALK_max:.2f} µmol/kg")

# ==================================================
# Allocate output array
# ==================================================
dpH_max = np.zeros((len(ALK_vals), len(DIC_vals)))

# ==================================================
# Loop over background ALK–DIC phase space
# ==================================================
for i, ALK_bg in enumerate(ALK_vals):
    for j, DIC_bg in enumerate(DIC_vals):

        # ---------- Background pH ----------
        bg = pyco2.sys(
            par1=ALK_bg,
            par2=DIC_bg,
            par1_type=1,      # ALK
            par2_type=2,      # DIC
            salinity=S,
            temperature=T,
            pressure=P,
            total_phosphate=PO4,
            total_silicate=Si
        )

        # ---------- Perturbed pH (ALK + max anomaly) ----------
        pert = pyco2.sys(
            par1=ALK_bg + dALK_max,
            par2=DIC_bg,
            par1_type=1,
            par2_type=2,
            salinity=S,
            temperature=T,
            pressure=P,
            total_phosphate=PO4,
            total_silicate=Si
        )

        dpH_max[i, j] = pert["pH_total"]

# ==================================================
# Phase-space contour plot
# ==================================================
DIC_grid, ALK_grid = np.meshgrid(DIC_vals, ALK_vals)

plt.figure(figsize=(7, 5), dpi=300)

cs = plt.contourf(
    DIC_grid,
    ALK_grid,
    dpH_max,
    levels=20,
    cmap="viridis"
)

cbar = plt.colorbar(cs)
cbar.set_label("Maximum ΔpH")
# ==================================================
# Overlay observed and modeled points
# ==================================================

# Observed values
ALK_obs = 2345
DIC_obs = 2145

# Modeled values
ALK_mod = 2280
DIC_mod = 2100

# Plot points
plt.plot(
    DIC_obs, ALK_obs,
    marker='o',
    markersize=8,
    markeredgecolor='k',
    markerfacecolor='white',
    linestyle='none',
    label='Observed'
)

plt.plot(
    DIC_mod, ALK_mod,
    marker='^',
    markersize=8,
    markeredgecolor='k',
    markerfacecolor='red',
    linestyle='none',
    label='Modeled'
)

# Optional text labels (comment out if too busy)
plt.text(DIC_obs + 3, ALK_obs + 3, 'Observed', fontsize=8)
plt.text(DIC_mod + 3, ALK_mod + 3, 'Modeled', fontsize=8)

# Ensure legend shows points
#plt.legend(frameon=True)


plt.xlabel("Background DIC (µmol kg⁻¹)")
plt.ylabel("Background Alkalinity (µmol kg⁻¹)")
plt.title("Maximum pH Excursion from Maximum ALK Anomaly")

plt.tight_layout()
plt.show()


In [ ]:
import xarray as xr
import numpy as np

# Load ROMS output using your pattern
roms_output = ROMSOutput(
    grid=grid,
    path=[
        model_bgc_path,
    ],
    use_dask=True,
)

ds_trac = roms_output.regrid(depth_levels=target_depth_levels, var_names=["ALK", "DIC","ALK_ALT_CO2","DIC_ALT_CO2"])


In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.mpl.ticker as cticker
import numpy as np

# Target snapshot
t_snap = np.datetime64("2024-02-21T00:00:00")
depth_target = 2.0

# Select nearest time & depth
ds_snap = ds_trac.sel(
    time=t_snap,
    depth=depth_target,
    method="nearest"
)


In [ ]:
ALK_anom = ds_snap["ALK"] - ds_snap["ALK_ALT_CO2"]
DIC_anom = ds_snap["DIC"] - ds_snap["DIC_ALT_CO2"]


In [ ]:
fig = plt.figure(figsize=(14, 6))
gs = fig.add_gridspec(1, 2, wspace=0.08)

extent_upper_hval = [
    -21.6,   # lon min
    -21.37,  # lon max
    64.34,   # lat min
    64.42    # lat max
]
proj = ccrs.Mercator()
data_crs = ccrs.PlateCarree()

# ==========================
# Panel 1 — ALK anomaly
# ==========================
ax1 = fig.add_subplot(gs[0, 0], projection=proj)
ax1.set_extent(extent_upper_hval, crs=data_crs)

cf1 = ax1.contourf(
    ds_trac["lon"], ds_trac["lat"],
    ALK_anom,
    levels=200,
    cmap="OrRd",vmin=0, vmax=100,
    transform=data_crs,
    extend="both"
)

ax1.set_title("ALK anomaly (ALK − ALK_ALT_CO2)\n2 m depth, 2024-02-21 00:00")
#ax1.set_xticks([-22.2, -22.0, -21.8, -21.6, -21.4], crs=data_crs)
#ax1.set_yticks([64.26, 64.30, 64.34, 64.38, 64.42], crs=data_crs)
ax1.xaxis.set_major_formatter(cticker.LongitudeFormatter())
ax1.yaxis.set_major_formatter(cticker.LatitudeFormatter())

cbar1 = fig.colorbar(cf1, ax=ax1, shrink=0.85, pad=0.03)
cbar1.set_label("ALK anomaly (µmol kg⁻¹)")

# ==========================
# Panel 2 — DIC anomaly
# ==========================
ax2 = fig.add_subplot(gs[0, 1], projection=proj)
ax2.set_extent(extent_upper_hval, crs=data_crs)

cf2 = ax2.contourf(
    ds_trac["lon"], ds_trac["lat"],
    DIC_anom,
    levels=31,
    cmap="OrRd", 
    transform=data_crs,
    extend="both"
)

ax2.set_title("DIC anomaly (DIC − DIC_ALT_CO2)\n2 m depth, 2024-02-21 00:00")
#ax2.set_xticks([-22.2, -22.0, -21.8, -21.6, -21.4], crs=data_crs)
#ax2.set_yticks([64.26, 64.30, 64.34, 64.38, 64.42], crs=data_crs)
ax2.xaxis.set_major_formatter(cticker.LongitudeFormatter())
ax2.yaxis.set_major_formatter(cticker.LatitudeFormatter())

cbar2 = fig.colorbar(cf2, ax=ax2, shrink=0.85, pad=0.03)
cbar2.set_label("DIC anomaly (µmol kg⁻¹)")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# --------------------------------------------------
# User settings
# --------------------------------------------------
depth_target = 2.0   # meters

# --------------------------------------------------
# Select depth
# --------------------------------------------------
ds_2m = ds_trac.sel(depth=depth_target, method="nearest")

# --------------------------------------------------
# Compute anomalies
# --------------------------------------------------
ALK_anom = ds_2m["ALK"] - ds_2m["ALK_ALT_CO2"]
DIC_anom = ds_2m["DIC"] - ds_2m["DIC_ALT_CO2"]

# --------------------------------------------------
# Compute grid-cell area weights (lat/lon grid)
# --------------------------------------------------
R = 6371e3  # Earth radius (m)

lat_rad = np.deg2rad(ds_2m.lat)
lon_rad = np.deg2rad(ds_2m.lon)

dlat = float(np.abs(lat_rad[1] - lat_rad[0]))
dlon = float(np.abs(lon_rad[1] - lon_rad[0]))

cell_area = (
    R**2
    * dlat
    * dlon
    * np.cos(lat_rad)
)

area = xr.DataArray(
    cell_area,
    coords={"lat": ds_2m.lat},
    dims=("lat",)
).broadcast_like(ALK_anom.isel(time=0))

# --------------------------------------------------
# Area-weighted sums
# --------------------------------------------------
ALK_sum = (ALK_anom * area).sum(dim=("lat", "lon"), skipna=True)
DIC_sum = (DIC_anom * area).sum(dim=("lat", "lon"), skipna=True)

# --------------------------------------------------
# Ratio
# --------------------------------------------------
ratio =  DIC_sum / ALK_sum 
ratio = ratio.where(np.isfinite(ratio))

# --------------------------------------------------
# Plot
# --------------------------------------------------
plt.figure(figsize=(14, 5))

plt.plot(
    ratio.time,
    ratio,
    color="darkmagenta",
    linewidth=2,
)

plt.title(
    "Area-weighted ratio of DIC anomaly to ALK anomaly\n"
    "2 m depth"
)
plt.xlabel("Time")
plt.ylabel("Σ(DIC anomaly) / Σ(ALK anomaly) ")
plt.grid(True)

plt.tight_layout()
plt.show()
